<a href="https://colab.research.google.com/github/minyi-k03/Large-Language-Model-LLM-/blob/Fine-Tuning/Llama2_fine_tuning_korquad.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Llama 2를 KorQuad 데이터셋에 맞게 fine-tuning 하기
### References:
1.   https://www.youtube.com/watch?v=LslC2nKEEGU
2.   https://colab.research.google.com/drive/1JDnGJbxT8fSqwnXY8J-XFo73AtiSuQMe?usp=sharing

In [1]:
!nvidia-smi

Mon Jan 19 07:20:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## AutoTrain Advanced
https://github.com/huggingface/autotrain-advanced

In [2]:
import os

print("🛑 설치 시작. 절대 끄지 마세요.")

# 1. 기존 찌꺼기 제거
!pip uninstall -y transformers autotrain-advanced peft triton bitsandbytes accelerate > /dev/null 2>&1

# 2. 필수 라이브러리 설치 (최신 버전 사용)
# autotrain-advanced는 최신 transformers에 의존하므로 전부 최신으로 맞춥니다.
!pip install -U autotrain-advanced
!pip install -U bitsandbytes
!pip install -U transformers peft accelerate torch torchvision torchaudio

print("✅ 설치 완료. 상단 메뉴 [런타임] -> [세션 다시 시작]을 누르세요.")

🛑 설치 시작. 절대 끄지 마세요.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.4/13.4 MB 109.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 9.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.2/78.2 kB 8.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 4.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.6/51.6 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 341.3/341.3 kB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.4/336.4 kB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 269.9/269.9 kB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.5/225.5 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 14.7 MB/s eta 0:00:00
  Attempting uninstall: bitsandbytes
    Found existing installation: bitsandbytes 0.45.0
    Uninstalling bitsandbytes-0.45.0:
      Successfully uninstalled bitsandbytes-0.45.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
autotrain-advanced 0.8.36 requires bitsandbytes==0.45.0; sys_platform == "linux", but you have bitsandbytes 0.49.1 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 148.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━

## PyTorch 업데이트

In [ ]:
!autotrain setup --update-torch

> INFO    Installing latest transformers@main
> INFO    Successfully installed latest transformers
> INFO    Installing latest peft@main
> INFO    Successfully installed latest peft
> INFO    Installing latest diffusers@main
> INFO    Successfully installed latest diffusers
> INFO    Installing latest trl@main
> INFO    Successfully installed latest trl
> INFO    Installing latest xformers
> INFO    Successfully installed latest xformers
> INFO    Installing latest PyTorch
> INFO    Successfully installed latest PyTorch


## Korquad 데이터셋에 맞게 Llama2 Fine-Tuning

- autotrain 설정값: https://github.com/huggingface/autotrain-advanced/blob/f1367b590dfc53d240e9684779991da540590386/src/autotrain/cli/run_llm.py#L21 (**과거 버전[0.6.35]**)
- autotrain 설정값: https://github.com/huggingface/autotrain-advanced/blob/main/src/autotrain/cli/run_llm.py#L17 (**최신 버전[0.6.80]**)


In [1]:
# 1. 기존 폴더 정리
!rm -rf llama2-korquad-finetuning-da

# 2. 학습 시작 (GPU 경로 강제 주입 모드)
# 아래 명령어 한 줄이 핵심입니다. 절대 수정하지 마세요.
!LD_LIBRARY_PATH=/usr/lib64-nvidia:/usr/local/cuda/lib64:$LD_LIBRARY_PATH \
 autotrain llm --train \
    --project-name "llama2-korquad-finetuning-da" \
    --model "TinyPixel/Llama-2-7B-bf16-sharded" \
    --data-path "korquad_prompt_da" \
    --text-column "text" \
    --peft \
    --quantization "int4" \
    --lr 1e-4 \
    --batch-size 8 \
    --epochs 10 \
    --trainer sft \
    --model_max_length 256

INFO     | 2026-01-19 07:54:39 | autotrain.cli.run_llm:run:136 - Running LLM
WARNING  | 2026-01-19 07:54:39 | autotrain.trainers.common:__init__:286 - Parameters supplied but not used: train, deploy, backend, config, inference, func, version
INFO     | 2026-01-19 07:54:39 | autotrain.backends.local:create:20 - Starting local training...
INFO     | 2026-01-19 07:54:39 | autotrain.commands:launch_command:514 - ['accelerate', 'launch', '--num_machines', '1', '--num_processes', '1', '--mixed_precision', 'no', '-m', 'autotrain.trainers.clm', '--training_config', 'llama2-korquad-finetuning-da/training_params.json']
INFO     | 2026-01-19 07:54:39 | autotrain.commands:launch_command:515 - {'model': 'TinyPixel/Llama-2-7B-bf16-sharded', 'project_name': 'llama2-korquad-finetuning-da', 'data_path': 'korquad_prompt_da', 'train_split': 'train', 'valid_split': None, 'add_eos_token': True, 'block_size': 1024, 'model_max_length': 256, 'padding': 'right', 'trainer': 'sft', 'use_flash_attention_2': False

# 학습결과 zip 파일로 압축후 다운로드

In [2]:
import zipfile
import shutil
from google.colab import files

# 압축할 폴더 이름
#folder_name = "llama2-korquad-finetuning"    # Data Augmentation 적용 x
folder_name = "llama2-korquad-finetuning-da"  # Data Augmentation 적용 o

# 생성될 ZIP 파일 이름
#zip_file_name = "llama2-korquad-finetuning.zip"  # Data Augmentation 적용 x
zip_file_name = "llama2-korquad-finetuning-da.zip" # Data Augmentation 적용 o

# 폴더를 ZIP 파일로 압축
shutil.make_archive(zip_file_name[:-4], 'zip', folder_name)

# ZIP 파일을 로컬로 다운로드
files.download(zip_file_name)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Llama-2 Fine-Tuning 모델 성능 테스트

In [1]:
import torch
from peft import PeftModel, PeftConfig
from transformers import AutoModelForCausalLM, AutoTokenizer

# 1. 경로 설정 (학습 결과 폴더 이름)
peft_model_id = "llama2-korquad-finetuning-da"

# 2. 기본 모델(Llama-2) + 토크나이저 불러오기
config = PeftConfig.from_pretrained(peft_model_id)
base_model = AutoModelForCausalLM.from_pretrained(
    "TinyPixel/Llama-2-7B-bf16-sharded",
    return_dict=True,
    torch_dtype=torch.float16,
    device_map='auto'
)
tokenizer = AutoTokenizer.from_pretrained(config.base_model_name_or_path)

# 3. 우리가 만든 학습 결과(Adapter) 합체!
model = PeftModel.from_pretrained(base_model, peft_model_id)

# 4. 질문 던져보기 (테스트)
def ask_model(question):
    # 프롬프트 형식은 학습시킬 때 썼던 형식과 맞춰주는 게 좋습니다.
    prompt = f"질문: {question}\n답변:"
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    # 생성
    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs["input_ids"],
            max_new_tokens=50,  # 답변 길이
            do_sample=True,     # 창의적 생성
            top_p=0.9,
            temperature=0.7
        )

    # 결과 해독
    print(tokenizer.decode(outputs[0], skip_special_tokens=True))

# 실행!
print("🤖 모델 로딩 완료! 테스트를 시작합니다.")
ask_model("대한민국 수도는?")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/14 [00:00<?, ?it/s]

You are using the default legacy behaviour of the <class 'transformers.models.llama.tokenization_llama_fast.LlamaTokenizerFast'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565 - if you loaded a llama tokenizer from a GGUF file you can ignore this message.


🤖 모델 로딩 완료! 테스트를 시작합니다.
질문: 고용민이 누구야?
답변: 임종석
